Steps for perfroming training

1. Import the modules
2. Perform feature engineering on the data
3. Segregate the data
4. Perform evaluation through all of the algo, and print the evaluations

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
# Modelling
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings

In [4]:
from pathlib import Path

data_path = Path('notebook/data/stud.csv')
if not data_path.exists():
    data_path = Path('data/stud.csv')

df = pd.read_csv(data_path)

# X = df.drop(columns = ["math_score"], axis = 1)

X = df.drop(columns=['math_score'])

y = df["math_score"]


In [5]:
# Reset X to the original feature dataframe before preprocessing
X = df.drop(columns=['math_score']).copy()
numerical_features = X.select_dtypes(exclude=['object', 'string']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'string']).columns.tolist()

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("OneHotEncoder", OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ("StandardScaler", StandardScaler(), numerical_features)
    ],
    remainder='drop'
)

X = preprocessor.fit_transform(X)

In [6]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape,X_test.shape

((800, 19), (200, 19))

In [7]:
def evaluate(true, predict):

    mse = mean_squared_error(true, predict)
    mae = mean_absolute_error(true, predict)
    rmse = np.sqrt(mean_squared_error(true, predict))
    r2 = r2_score(true, predict)

    return mae, rmse, r2



models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(), 
    "CatBoosting Regressor": CatBoostRegressor(verbose=False),
    "AdaBoost Regressor": AdaBoostRegressor()
}


models_list = []
r2_list = []


for i in range(len(list(models))):

    model = list(models.values())[i]

    model.fit(X_train, y_train)


    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)



    train_mae, train_rmse, train_r2 = evaluate(y_train, y_train_pred)
    test_mae, test_rmse, test_r2 = evaluate(y_test, y_test_pred)


    print(list(models.keys())[i])

    models_list.append(list(models.keys())[i])

    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(train_mae))
    print("- R2 Score: {:.4f}".format(train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(test_mae))
    print("- R2 Score: {:.4f}".format(test_r2))

    r2_list.append(test_r2)

    print('='*35)
    print('\n')


Linear Regression
Model performance for Training set
- Root Mean Squared Error: 5.3231
- Mean Absolute Error: 4.2667
- R2 Score: 0.8743
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 5.3940
- Mean Absolute Error: 4.2148
- R2 Score: 0.8804


Lasso
Model performance for Training set
- Root Mean Squared Error: 6.5938
- Mean Absolute Error: 5.2063
- R2 Score: 0.8071
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 6.5197
- Mean Absolute Error: 5.1579
- R2 Score: 0.8253


Ridge
Model performance for Training set
- Root Mean Squared Error: 5.3233
- Mean Absolute Error: 4.2650
- R2 Score: 0.8743
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 5.3904
- Mean Absolute Error: 4.2111
- R2 Score: 0.8806


K-Neighbors Regressor
Model performance for Training set
- Root Mean Squared Error: 5.7077
- Mean Absolute Error: 4.5167
- R2 Score: 0.8555
-----------------------

In [8]:
pd.DataFrame(list(zip(models_list,r2_list)), columns=["Models name", "R2_Score"]).sort_values(by=["R2_Score"], ascending=False)

,Models name,R2_Score
2,Ridge,0.880593
0,Linear Regression,0.880433
7,CatBoosting Regressor,0.851632
5,Random Forest Regressor,0.851535
8,AdaBoost Regressor,0.846609
6,XGBRegressor,0.827797
1,Lasso,0.825320
3,K-Neighbors Regressor,0.783813
4,Decision Tree,0.747492


In [10]:
from sklearn.linear_model import LinearRegression
lin_model = LinearRegression(fit_intercept=False)
lin_model = lin_model.fit(X_train, y_train)
y_pred = lin_model.predict(X_test) 
variancy_in_data = r2_score(y_test, y_pred)

print("The actual variance  of data that the model predicts well is about : ",variancy_in_data)

The actual variance  of data that the model predicts well is about :  0.8804332983749564
